# SLM-G2P v11 フルパイプライン（データセット構築 → 学習 → 評価予測生成）

論文の全結果をこのノートブック1本で再現する。ランタイム→**A100 GPU**。実行前に、リポジトリ `data/heteronym_eval_set_v1.csv` を
作業ディレクトリ（Colabなら `/content`）にアップロードしてから、すべてのセルを実行。

| フェーズ | 内容 | 目安時間 |
|---|---|---|
| 構築 | 成分(a)〜(h)・約243万行＋SHA-1汚染監査 | 初回約2時間（キャッシュ後は約30分） |
| 学習 | sarashina2.2-1b + LoRA r=16, 3エポック | 約29〜33時間（切断時は再実行で自動再開） |
| 評価 | 文分割＋カナ制約＋ビーム4で全6ベンチの予測生成 | 約3〜5時間 |

採点はローカルで `eval/` のスクリプトを使う（README参照）。
**終了後に共有/保存**: `results_v11/final`（アダプタ）と `results_v11_beam/`（全予測）。


In [ ]:
import torch, os
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
WORK = "/content/drive/MyDrive/research/g2p_run"
os.makedirs(f"{WORK}/cache", exist_ok=True)
gpu = torch.cuda.get_device_name(0)
BATCH, GRAD_ACCUM = (16, 2)
print("GPU:", gpu, "| batch:", BATCH, "x", GRAD_ACCUM)

In [ ]:
%%capture
!pip uninstall -y -q torchao
!pip install -q "transformers>=4.46" datasets peft accelerate jiwer pyopenjtalk tiktoken sentencepiece protobuf sudachipy==0.6.11 sudachidict_core

In [ ]:
# ===== 設定（v8） =====
import os as _os
_os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
_os.environ["TRANSFORMERS_VERBOSITY"] = "error"
try:
    import datasets as _dts; _dts.disable_progress_bars()
except Exception:
    pass
SEED = 42
EPOCHS = 3
VER = "v11"
BULK_VER = "v8"              # bulk(a)(b)はレシピ不変のためv8キャッシュを再利用
LR = 5e-5
WARMUP_RATIO = 0.03
LORA_R = 16
MAX_LEN = 512
N_BULK = 1_600_000          # 合意フィルタ（助詞発音化版）通過文の採用数
N_DIGIT_BULK = 200_000      # 数字含み文（合意不要・pyojラベル）の採用数
CORR_OVERSAMPLE = 2         # 青空: pyoj矛盾ルビ文（訂正信号）の倍率
AOZORA_COMMIT = "0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65"
HUMAN_OVERSAMPLE = 3        # KWDLC/ROHAN（人手読み文）の倍率
JSUT_OVERSAMPLE = 3         # jsut-label（人手発音注釈）の倍率
JSUT_COMMIT = "1978271ca6212e1ea742da8f149160f5679e8971"  # sarulab-speech/jsut-label
N_DICT = 300_000            # SudachiDict一意読み語の採用数
NUM_OVERSAMPLE = 3          # ルール生成データの倍率
RULE_N_PER = 2000
KWDLC_COMMIT = "c6ae49d29eca"
ROHAN_COMMIT = "98082649e1e0"
SUDACHI_RAW_VER = "20260723"
PARAKEET = "Parakeet-Inc/joyo-kanji-yomi-benchmark-parakeet"   # split="test", MIT
G2PBENCH_URL = ("https://raw.githubusercontent.com/filmapp/ja-tts-g2p-bench/"
                "12182d464d4b5df82b829369974bf72d24a6f0cf/data/items_read_bench_v1.jsonl")  # MIT
MODEL = "sbintuitions/sarashina2.2-1b-instruct-v0.1"
SYSTEM = ("あなたは日本語テキストの読み推定器です。与えられた文の読みを、カタカナのみで1行で出力してください。"
          "発音通りに書くこと: 助詞の「は」は「ワ」、「へ」は「エ」、「を」は「オ」。数字や記号も読みをカタカナで書くこと。")


In [ ]:
# ===== 数字・助数詞・英字のルール生成データ（決定的・AI注釈なし） =====
"""読みはすべてテーブル＋規則でプログラムが算出する（検証可能）。"""
import json
import random

DIG = {"0": "ゼロ", "1": "イチ", "2": "ニ", "3": "サン", "4": "ヨン",
       "5": "ゴ", "6": "ロク", "7": "ナナ", "8": "ハチ", "9": "キュー"}

def digits_read(s):
    return "".join(DIG[c] for c in s)

_U = ["", "イチ", "ニ", "サン", "ヨン", "ゴ", "ロク", "ナナ", "ハチ", "キュー"]
_H = ["", "ヒャク", "ニヒャク", "サンビャク", "ヨンヒャク", "ゴヒャク",
      "ロッピャク", "ナナヒャク", "ハッピャク", "キューヒャク"]
_S = ["", "セン", "ニセン", "サンゼン", "ヨンセン", "ゴセン",
      "ロクセン", "ナナセン", "ハッセン", "キューセン"]

def pos_read(n):
    """1〜9999の位取り読み"""
    assert 1 <= n <= 9999
    s = _S[n // 1000] + _H[n // 100 % 10]
    t = n // 10 % 10
    if t:
        s += ("ジュー" if t == 1 else _U[t] + "ジュー")
    return s + _U[n % 10]

def year_read(n):
    return (pos_read(n - 4) + "ヨ" if n % 10 == 4 else pos_read(n)) + "ネン"

COUNTERS = {
 "本": "イッポン ニホン サンボン ヨンホン ゴホン ロッポン ナナホン ハッポン キューホン ジュッポン",
 "匹": "イッピキ ニヒキ サンビキ ヨンヒキ ゴヒキ ロッピキ ナナヒキ ハッピキ キューヒキ ジュッピキ",
 "分": "イップン ニフン サンプン ヨンプン ゴフン ロップン ナナフン ハップン キューフン ジュップン",
 "階": "イッカイ ニカイ サンガイ ヨンカイ ゴカイ ロッカイ ナナカイ ハチカイ キューカイ ジュッカイ",
 "歳": "イッサイ ニサイ サンサイ ヨンサイ ゴサイ ロクサイ ナナサイ ハッサイ キューサイ ジュッサイ",
 "冊": "イッサツ ニサツ サンサツ ヨンサツ ゴサツ ロクサツ ナナサツ ハッサツ キューサツ ジュッサツ",
 "頭": "イットー ニトー サントー ヨントー ゴトー ロクトー ナナトー ハットー キュートー ジュットー",
 "個": "イッコ ニコ サンコ ヨンコ ゴコ ロッコ ナナコ ハッコ キューコ ジュッコ",
}
COUNTERS = {k: v.split() for k, v in COUNTERS.items()}
WARI = [None] + "イチワリ ニワリ サンワリ ヨンワリ ゴワリ ロクワリ ナナワリ ハチワリ キューワリ".split()
BU = [None] + "イチブ ニブ サンブ ヨンブ ゴブ ロクブ ナナブ ハチブ キューブ".split()
RIN = [None] + "イチリン ニリン サンリン ヨンリン ゴリン ロクリン ナナリン ハチリン キューリン".split()

ALPHA = {c: r for c, r in zip("ABCDEFGHIJKLMNOPQRSTUVWXYZ",
    ["エー","ビー","シー","ディー","イー","エフ","ジー","エイチ","アイ","ジェー","ケー","エル","エム",
     "エヌ","オー","ピー","キュー","アール","エス","ティー","ユー","ブイ","ダブリュー","エックス","ワイ","ゼット"])}

def alpha_read(s):
    return "".join(ALPHA[c] for c in s)

def gen(seed=42, n_per=400):
    rng = random.Random(seed)
    rows = []
    def add(text, kana):
        rows.append({"text": text, "kana": kana})
    for _ in range(n_per):
        a, b, c = (str(rng.randint(0, 9)) for _ in range(3))
        mid = "".join(str(rng.randint(0, 9)) for _ in range(4))
        last = "".join(str(rng.randint(0, 9)) for _ in range(4))
        head = f"0{a}0"
        add(f"電話番号は{head}の{mid}の{last}です。",
            f"デンワバンゴーワ{digits_read(head)}ノ{digits_read(mid)}ノ{digits_read(last)}デス")
        pin = "".join(str(rng.randint(0, 9)) for _ in range(4))
        add(f"暗証番号は{pin}です。", f"アンショーバンゴーワ{digits_read(pin)}デス")
        code = "".join(str(rng.randint(0, 9)) for _ in range(3))
        add(f"会員番号は{code}番です。", f"カイインバンゴーワ{digits_read(code)}バンデス")
        y = rng.randint(1900, 2029)
        add(f"西暦{y}年に設立された。", f"セイレキ{year_read(y)}ニセツリツサレタ")
        w, bu, rin = rng.randint(1, 9), rng.randint(1, 9), rng.randint(1, 9)
        add(f"打率は{w}割{bu}分{rin}厘だった。", f"ダリツワ{WARI[w]}{BU[bu]}{RIN[rin]}ダッタ")
        d1, d2 = rng.randint(0, 9), rng.randint(1, 9)
        i = rng.randint(10, 39)
        add(f"気温は{i}.{d2}度まで上がった。",
            f"キオンワ{pos_read(i)}テン{DIG[str(d2)]}ドマデアガッタ")
        n = rng.randint(1, 10)
        tmpl = [("机の上に鉛筆が{n}本ある。", "ツクエノウエニエンピツガ{y}アル", "本"),
                ("公園で猫を{n}匹見かけた。", "コーエンデネコオ{y}ミカケタ", "匹"),
                ("その本を{n}冊注文した。", "ソノホンオ{y}チューモンシタ", "冊"),
                ("ビルの{n}階で働いている。", "ビルノ{y}デハタライテイル", "階"),
                ("会議は{n}分遅れて始まった。", "カイギワ{y}オクレテハジマッタ", "分"),
                ("牧場で牛を{n}頭飼っている。", "ボクジョーデウシオ{y}カッテイル", "頭"),
                ("箱にりんごが{n}個入っている。", "ハコニリンゴガ{y}ハイッテイル", "個"),
                ("息子は{n}歳になった。", "ムスコワ{y}ニナッタ", "歳")]
        t, k, cnt = tmpl[rng.randrange(len(tmpl))]
        add(t.format(n=n), k.format(y=COUNTERS[cnt][n - 1]))
        # 英字・型番（pyopenjtalkが素通しする穴をルールで埋める）
        L = "".join(rng.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in range(rng.randint(2, 3)))
        num = "".join(str(rng.randint(0, 9)) for _ in range(3))
        add(f"型番は{L}{num}です。", f"カタバンワ{alpha_read(L)}{digits_read(num)}デス")
        L2 = "".join(rng.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in range(3))
        add(f"この組織は{L2}と略される。", f"コノソシキワ{alpha_read(L2)}トリャクサレル")
    return rows


NUM_RULE_ROWS = [json.dumps(r, ensure_ascii=False) for r in gen(seed=SEED, n_per=RULE_N_PER)]
print('number/alpha-rule rows:', len(NUM_RULE_ROWS))

In [ ]:
# ===== 共通: 正規化・整形式・除外セット・Sudachi（助詞発音化） =====
import csv, hashlib, json, re, urllib.request
from functools import lru_cache
import pyopenjtalk
from sudachipy import dictionary as _sd, tokenizer as _st
SDICT = _sd.Dictionary(); sud = SDICT.create(); SMODE = _st.Tokenizer.SplitMode.C
KATA_H2K = str.maketrans({chr(c): chr(c+0x60) for c in range(0x3041,0x3097)})
KATA_K2H = str.maketrans({chr(c+0x60): chr(c) for c in range(0x3041,0x3097)})
def is_test_work(work):
    return int(hashlib.sha1(work.encode()).hexdigest(),16) % 100 < 3
STRIP = re.compile(r"[。．，！？!?・\s「」『』（）()〔〕［］…‥－—−‐―〜~]")
WELLFORMED = re.compile(r"^[ァ-ヶー、]+$")
DIGIT = re.compile(r"[0-9０-９]")
HAS_KANJI = re.compile(r"[一-龠]")
urllib.request.urlretrieve("https://raw.githubusercontent.com/CyberAgentAILab/jvs_nonpara_kana/main/jvs_nonpara_kana.csv","jvs_nonpara_kana.csv")
urllib.request.urlretrieve("https://raw.githubusercontent.com/CyberAgentAILab/jvs_nonpara_kana/main/eval_cer.py","eval_cer.py")
EXCLUDE = set(r[1] for r in csv.reader(open("jvs_nonpara_kana.csv",encoding="utf-8")))
_V = {}
for _row, _v in [("アカガサザタダナハバパマヤラワャァ","a"),("イキギシジチヂニヒビピミリィ","i"),
                 ("ウクグスズツヅヌフブプムユルュゥヴ","u"),("エケゲセゼテデネヘベペメレェ","e"),
                 ("オコゴソゾトドノホボポモヨロヲョォ","o")]:
    for _ch in _row: _V[_ch] = _v
def pron_norm(s):
    # 発音等価正規化（比較専用。ラベルには適用しない）
    s = s.replace("ヲ","オ").replace("ヂ","ジ").replace("ヅ","ズ")
    out = []
    for ch in s:
        if out:
            pv = _V.get(out[-1])
            if ch == "ー" and pv:
                out.append({"a":"ア","i":"イ","u":"ウ","e":"エ","o":"オ"}[pv]); continue
            if (ch == "ウ" and pv in ("o","u")) or (ch == "イ" and pv in ("e","i")):
                out.append({"o":"オ","u":"ウ","e":"エ","i":"イ"}[pv]); continue
        out.append(ch)
    return "".join(out).replace("トユウ","トイウ")
def norm_kana(s):
    return STRIP.sub("", s).replace("ヲ","オ")
def sud_read(text):
    # Sudachi読み。助詞は/へは発音（ワ/エ）に直す
    out = []
    for m in sud.tokenize(text, SMODE):
        r = m.reading_form()
        if m.part_of_speech()[0] == "助詞":
            if m.surface() == "は": r = "ワ"
            elif m.surface() == "へ": r = "エ"
        out.append(r)
    return "".join(out)
@lru_cache(maxsize=None)
def attested(w):
    # 語wの辞書「見出し語」読み集合（厳密版: 決定的な集合所属テスト、人手・AI判断なし）
    # SudachiDict(20260723)またはnaist-jdic(1.11)にwが見出しとして載っている読みのみ。
    # 解析器が複数形態素から合成した読みは「載っている」ではないため採用しない
    rs = set()
    try:
        for x in SDICT.lookup(w):
            if x.surface() == w: rs.add(x.reading_form())      # SudachiDict見出し
    except Exception:
        pass
    toks = sud.tokenize(w, SMODE)
    if len(toks) == 1 and toks[0].surface() == w:
        rs.add(toks[0].reading_form())                          # Sudachi見出し(1形態素)
    fr = pyopenjtalk.run_frontend(w)
    if len(fr) == 1 and fr[0]["string"] == w:
        rs.add(fr[0]["pron"].replace("’", ""))                  # naist-jdic見出し(1形態素)
    return frozenset(pron_norm(r) for r in rs if r)
# 外部ベンチ（評価用）: ja-tts-g2p-bench 160問（コミット固定）
urllib.request.urlretrieve(G2PBENCH_URL, "items_read_bench_v1.jsonl")
G2PBENCH = [json.loads(l) for l in open("items_read_bench_v1.jsonl", encoding="utf-8")]
print("exclude:", len(EXCLUDE), "| g2pbench:", len(G2PBENCH))
# 困難セット300文（評価用・訓練から除外するためここで定義）
import hashlib
HARD_SET_CSV = "heteronym_eval_set_v1.csv"   # このノートブックと同じディレクトリに配置（リポジトリ data/ に同梱）
try:
    _hs_rows = list(csv.reader(open(HARD_SET_CSV, encoding="utf-8")))[1:]
except FileNotFoundError:
    raise FileNotFoundError("heteronym_eval_set_v1.csv を作業ディレクトリにアップロードしてください（リポジトリの data/ にあります）")
HARD_SET = [{"id": r[0], "text": r[1]} for r in _hs_rows]
assert len(HARD_SET) == 300


In [ ]:
# ===== データ構築①: KWDLC（人手読み・train分割のみ）と ROHAN4600（約5分） =====
import glob
!git clone -q --filter=blob:none https://github.com/ku-nlp/KWDLC /content/KWDLC
!cd /content/KWDLC && git checkout -q c6ae49d29eca
def parse_knp(fn):
    sents = []; surf = []; yomi = []
    for l in open(fn, encoding="utf-8"):
        if l.startswith(("#","*","+")): continue
        if l.startswith("EOS"):
            if surf: sents.append(("".join(surf), "".join(yomi)))
            surf, yomi = [], []
            continue
        f = l.split(" ")
        if len(f) < 4: continue
        s, y, pos = f[0], f[1], f[3]
        if pos == "助詞":
            if s == "は": y = "わ"
            elif s == "へ": y = "え"
        surf.append(s); yomi.append(y)
    return sents
ids = {sp: set(l.strip() for l in open(f"/content/KWDLC/id/{sp}.id")) for sp in ("train","test")}
kwdlc = {"train": [], "test": []}
for fn in glob.glob("/content/KWDLC/knp/*/*.knp"):
    did = fn.split("/")[-1][:-4]
    sp = "train" if did in ids["train"] else ("test" if did in ids["test"] else None)
    if sp is None: continue
    for text, y in parse_knp(fn):
        kana = STRIP.sub("", y.translate(KATA_H2K)).replace("ヲ","オ")
        if DIGIT.search(text) or not HAS_KANJI.search(text) or not (6 <= len(text) <= 80): continue
        if text in EXCLUDE or not kana or not WELLFORMED.match(kana): continue
        kwdlc[sp].append((text, kana))
print("KWDLC train:", len(kwdlc["train"]), "/ test(新ベンチ・学習不使用):", len(kwdlc["test"]))
urllib.request.urlretrieve(f"https://raw.githubusercontent.com/mmorise/rohan4600/{ROHAN_COMMIT}/Rohan4600_transcript_utf8.txt", "rohan.txt")
rohan = []
for l in open("rohan.txt", encoding="utf-8"):
    l = l.strip()
    if not l or ":" not in l: continue
    t, k = l.split(":", 1)[1].rsplit(",", 1)
    plain = re.sub(r"\(([^()]*)\)", "", t)
    kana = STRIP.sub("", k).replace("ヲ","オ")
    if plain in EXCLUDE or not WELLFORMED.match(kana): continue
    rohan.append((plain, kana))
print("ROHAN:", len(rohan))

In [ ]:
# ===== データ構築①c: 成分(i) jsut-label（人手発音注釈・汚染除外・発話逸脱フィルタ） =====
import difflib, yaml
urllib.request.urlretrieve(
    f"https://raw.githubusercontent.com/sarulab-speech/jsut-label/{JSUT_COMMIT}/text_kana/basic5000.yaml",
    "jsut_basic5000.yaml")
_js = yaml.safe_load(open("jsut_basic5000.yaml", encoding="utf-8"))
import sys as _sys
if "." not in _sys.path: _sys.path.insert(0, ".")
from eval_cer import normalize_long_vowels
JS_STRIP = re.compile(r"[。．，！？!?・\s「」『』（）()〔〕［］…‥－—−‐―〜~、]")
JS_DIGIT = re.compile(r"[0-9０-９〇一二三四五六七八九十百千万億]")
def js_align(s):
    return normalize_long_vowels(JS_STRIP.sub("", s).replace("ヲ", "オ"))
def js_aligned_span(ref, hyp, s, e):
    n, m = len(ref), len(hyp)
    D = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): D[i][0] = i
    for j in range(m+1): D[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            D[i][j] = min(D[i-1][j]+1, D[i][j-1]+1, D[i-1][j-1]+(ref[i-1] != hyp[j-1]))
    i, j = n, m
    map_end = [m]*(n+1)
    while i > 0 or j > 0:
        map_end[i] = j
        if i > 0 and j > 0 and D[i][j] == D[i-1][j-1]+(ref[i-1] != hyp[j-1]): i, j = i-1, j-1
        elif i > 0 and D[i][j] == D[i-1][j]+1: i -= 1
        else: j -= 1
    map_end[0] = 0
    return hyp[map_end[s]:map_end[e]]
def _att_norm(w):
    return {pron_norm(js_align(r)) for r in attested(w)}
_jvs_basic_ids = set(r[0].split("_", 1)[1] for r in csv.reader(open("jvs_nonpara_kana.csv", encoding="utf-8"))
                     if "BASIC5000" in r[0])
# 全評価セットの文集合（ループ内でこの集合との一致を除外。Joyo/Parakeet例文との偶然一致1件も対象）
from datasets import load_dataset as _ld
_eval_texts = set(EXCLUDE) | set(h["text"] for h in HARD_SET) | set(t for t, _ in kwdlc["test"])
_eval_texts |= set(it["surface"] for it in G2PBENCH)
_eval_texts |= set(ex["normalized_text"] for ex in _ld("sbintuitions/joyo-kanji-yomi-benchmark", split="train"))
_eval_texts |= set(ex["text"] for ex in _ld(PARAKEET, split="test"))
JSUT_ROWS = []
n_excl = n_bad = n_agree = n_corr = n_drop = 0
for k, v in _js.items():
    if k in _jvs_basic_ids: n_excl += 1; continue
    text = v["text_level0"]
    if text in _eval_texts: n_excl += 1; continue     # 全評価セットとの完全一致を除外
    raw = (v.get("kana_level3") or v["kana_level0"]).translate(KATA_H2K)
    kana = norm_kana(raw)
    if not kana or not WELLFORMED.match(kana): n_bad += 1; continue
    j = js_align(raw)
    morphs = pyopenjtalk.run_frontend(text)
    prons = [js_align(m["pron"].replace("\u2019", "")) for m in morphs]
    p = "".join(prons)
    if pron_norm(p) == pron_norm(j):
        n_agree += 1
        JSUT_ROWS.append(json.dumps({"text": text, "kana": kana}, ensure_ascii=False)); continue
    offs = [0]
    for pr in prons: offs.append(offs[-1] + len(pr))
    ok = True
    for t, i1, i2, j1, j2 in difflib.SequenceMatcher(None, p, j).get_opcodes():
        if t == "equal": continue
        mi = max([x for x in range(len(morphs)) if offs[x] <= i1], default=0)
        me = min([x for x in range(len(morphs)) if offs[x+1] >= i2], default=len(morphs)-1)
        w = "".join(m["string"] for m in morphs[mi:me+1])
        jspan = js_aligned_span(p, j, offs[mi], offs[me+1])
        jn = pron_norm(jspan)
        if w in ("は", "へ") and jn in ("ワ", "エ"): continue
        if JS_DIGIT.search(w): continue
        if jn in _att_norm(w): continue
        ok = False; break
    if ok:
        n_corr += 1
        JSUT_ROWS.append(json.dumps({"text": text, "kana": kana}, ensure_ascii=False))
    else:
        n_drop += 1
print(f"jsut-label: 汚染除外{n_excl} 整形式落ち{n_bad} 一致採用{n_agree} 訂正採用{n_corr} "
      f"逸脱除外{n_drop} → 成分(i) {len(JSUT_ROWS)}文 x{JSUT_OVERSAMPLE}")
# ローカル実測の期待値: 汚染除外1840 一致2433 訂正382 逸脱345 → 2815文（SudachiDict版差で数件ずれ可）
# 最終セーフティ（構築ロジックが正しければ必ず0件）
_hits = [json.loads(r)["text"] for r in JSUT_ROWS if json.loads(r)["text"] in _eval_texts]
assert not _hits, f"jsut汚染検出: {_hits[:3]}"
print("jsut汚染チェックOK（jvs/Joyo/Parakeet/g2pbench/困難/KWDLC-testと交差なし）")


In [ ]:
# ===== データ構築②: Wikipedia合意フィルタ済みbulk 160万文（初回約2時間、キャッシュあれば即） =====
import os
CONSENSUS = f"{WORK}/cache/bulk_consensus_{BULK_VER}.tsv"
DIGITBULK = f"{WORK}/cache/bulk_digit_{BULK_VER}.tsv"
if not os.path.exists(CONSENSUS):
    from datasets import load_dataset
    # コンタミ防止: 全評価セットの文をbulkから完全一致除外
    EXTRA_EXCLUDE = set(it["surface"] for it in G2PBENCH)
    EXTRA_EXCLUDE |= set(ex["text"] for ex in load_dataset(PARAKEET, split="test"))
    EXTRA_EXCLUDE |= set(ex["normalized_text"] for ex in load_dataset("sbintuitions/joyo-kanji-yomi-benchmark", split="train"))
    EXTRA_EXCLUDE |= set(t for t, _ in kwdlc["train"]) | set(t for t, _ in kwdlc["test"])
    EXTRA_EXCLUDE |= set(t for t, _ in rohan)
    EXTRA_EXCLUDE |= set(h["text"] for h in HARD_SET)   # 困難セット（数字文bulkは合意排他が効かないため明示除外）
    print("extra exclude:", len(EXTRA_EXCLUDE))
    ALLOWED = re.compile(r"^[ぁ-んァ-ヶー一-龠々〆0-9０-９a-zA-Z、。・「」『』（）%％〜.,:：]+$")
    SPLIT = re.compile(r"(?<=[。！？])")
    ds = load_dataset("wikimedia/wikipedia","20231101.ja",split="train",streaming=True)
    seen=set(); n_seen=n_ok=n_dig=0
    with open(CONSENSUS,"w",encoding="utf-8") as f, open(DIGITBULK,"w",encoding="utf-8") as fd:
        for ex in ds:
            for s in SPLIT.split(ex["text"].replace("\n","")):
                s=s.strip()
                if not(10<=len(s)<=60) or not s.endswith("。"): continue
                if not HAS_KANJI.search(s) or not ALLOWED.match(s) or s in seen: continue
                seen.add(s); n_seen+=1
                if s in EXCLUDE or s in EXTRA_EXCLUDE: continue
                label = norm_kana(pyopenjtalk.g2p(s, kana=True))
                if not WELLFORMED.match(label): continue          # 整形式制約
                if DIGIT.search(s):
                    # 数字文: 合意不要・pyojラベル（合意フィルタは数字文を系統的に落とすため）
                    if n_dig < N_DIGIT_BULK:
                        fd.write(f"{s}\t{label}\n"); n_dig+=1
                else:
                    if n_ok < N_BULK and pron_norm(label) == pron_norm(norm_kana(sud_read(s))):
                        f.write(f"{s}\t{label}\n"); n_ok+=1
                if (n_ok+n_dig) % 100000 == 0:
                    print(f"consensus {n_ok}/{N_BULK}  digit {n_dig}/{N_DIGIT_BULK} (走査{n_seen})", flush=True)
                if n_ok >= N_BULK and n_dig >= N_DIGIT_BULK: break
            if n_ok >= N_BULK and n_dig >= N_DIGIT_BULK: break
    print(f"built: consensus {n_ok} + digit {n_dig} (走査{n_seen})")
else:
    print("cached:", sum(1 for _ in open(CONSENSUS,encoding="utf-8")), "+", sum(1 for _ in open(DIGITBULK,encoding="utf-8")))

In [ ]:
# ===== データ構築③: 青空文庫・辞書載録読みサブセット（キャッシュあれば抽出は即） =====
RUBY_ALL = f"{WORK}/cache/ruby_sentences.jsonl"
if not os.path.exists(RUBY_ALL):
    !git clone -q --filter=blob:none https://github.com/aozorahack/aozorabunko_text /content/aozora
    !cd /content/aozora && git checkout -q 0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65
    urllib.request.urlretrieve("https://www.aozora.gr.jp/index_pages/list_person_all_extended_utf8.zip","cat.zip")
    !unzip -qo cat.zip
    allowed=set()
    for row in csv.DictReader(open("list_person_all_extended_utf8.csv",encoding="utf-8")):
        if row["文字遣い種別"]!="新字新仮名": continue
        m=re.search(r"/files/(\d+_(?:ruby_|txt_)?\d*)\.zip", row.get("テキストファイルURL",""))
        if m: allowed.add(m.group(1))
    RUBY=re.compile(r"(?:｜([^《》｜\n]{1,20})|([一-龠々〆ヵヶ]+))《([ぁ-んゝゞー]+)》")
    NOTE=re.compile(r"［＃[^］]*］")
    from pathlib import Path
    seen=set(); n=0
    with open(RUBY_ALL,"w",encoding="utf-8") as out:
        for txt in Path("/content/aozora/cards").glob("*/files/*_ruby_*/*.txt"):
            fid=txt.parent.name
            if fid not in allowed: continue
            raw=txt.read_bytes().decode("shift_jis",errors="ignore")
            parts=raw.split("-------------------------------------------------------")
            body=(parts[2] if len(parts)>=3 else raw).split("底本：")[0]
            body=NOTE.sub("",body).replace("　","")
            for sent in re.split(r"(?<=[。！？])", body.replace("\n","").replace("\r","")):
                sent=sent.strip()
                if not(10<=len(sent)<=80) or "《" not in sent: continue
                spans=[]
                def repl(m):
                    base=m.group(1) or m.group(2)
                    spans.append([base, m.group(3).translate(KATA_H2K).replace("ゝ","").replace("ゞ","")])
                    return base
                plain=RUBY.sub(repl,sent)
                if not spans or "《" in plain or plain in seen or not(10<=len(plain)<=60): continue
                seen.add(plain)
                out.write(json.dumps({"text":plain,"spans":spans,"work":fid},ensure_ascii=False)+"\n")
                n+=1
    print("ruby sentences:",n)
else:
    print("cached ruby:", sum(1 for _ in open(RUBY_ALL,encoding="utf-8")))
# 載録フィルタ: 全spanが辞書載録読み・読み分け語（載録読み2種以上）を含む文のみ
AZ_ROWS=[]
n_corr=n_drop=n_skip=0
for l in open(RUBY_ALL,encoding="utf-8"):
    r=json.loads(l); text=r["text"]
    if text in EXCLUDE or is_test_work(r["work"]): continue
    atts=[attested(w) for w,rd in r["spans"]]
    if any(pron_norm(rd) not in A for (w,rd),A in zip(r["spans"],atts)): n_drop+=1; continue
    has_multi=any(len(A)>=2 for A in atts)
    sub,pos,ok=text,0,True
    for w,rd in r["spans"]:
        i=sub.find(w,pos)
        if i<0: ok=False; break
        h=rd.translate(KATA_K2H); sub=sub[:i]+h+sub[i+len(w):]; pos=i+len(h)
    if not ok: continue
    kana=norm_kana(pyopenjtalk.g2p(sub,kana=True))
    if not WELLFORMED.match(kana): continue
    pp=pron_norm(norm_kana(pyopenjtalk.g2p(text,kana=True)))
    corr=any(pron_norm(rd) not in pp for w,rd in r["spans"])
    # 採用条件: 読み分け語を含む（文脈対比の両側を学ぶ）または pyoj矛盾（訂正教材、縦波・無躾型の
    # 一意読み複合語の誤読修正を含む）。読み1つ・pyoj一致の文はラベルがpyoj出力と同一で情報ゼロのため不採用
    if not (has_multi or corr): n_skip+=1; continue
    row=json.dumps({"text":text,"kana":kana},ensure_ascii=False)
    AZ_ROWS.extend([row]*(CORR_OVERSAMPLE if corr else 1)); n_corr+=corr
print(f"青空載録サブセット: {len(AZ_ROWS)}行 (訂正信号x{CORR_OVERSAMPLE}: {n_corr}, 辞書外読みで除外: {n_drop}, 情報ゼロで除外: {n_skip})")

In [ ]:
# ===== データ構築③: SudachiDict一意読み語 + 全成分の混合 =====
TRAIN_FILE = f"{WORK}/cache/train_mix_{VER}.jsonl"
if not os.path.exists(TRAIN_FILE):
    # SudachiDict生CSV（公式S3・バージョン固定）から (表層→読み) を抽出
    from collections import defaultdict
    by_surface = defaultdict(set)
    for lex in ("small_lex", "core_lex"):
        urllib.request.urlretrieve(
            f"http://sudachi.s3-website-ap-northeast-1.amazonaws.com/sudachidict-raw/{SUDACHI_RAW_VER}/{lex}.zip",
            f"{lex}.zip")
        !unzip -qo {lex}.zip
        KANA_ONLY = re.compile(r"^[ァ-ヶー]+$")
        BADCH = re.compile(r"[0-9０-９a-zA-Zａ-ｚＡ-Ｚ]")
        for row in csv.reader(open(f"{lex}.csv", encoding="utf-8")):
            if len(row) < 12: continue
            s, r = row[0], row[11]
            if not (2 <= len(s) <= 8) or not HAS_KANJI.search(s) or BADCH.search(s): continue
            if not KANA_ONLY.match(r): continue
            by_surface[s].add(pron_norm(r))
    uniq = sorted((s, list(rs)[0]) for s, rs in by_surface.items() if len(rs) == 1)
    import random
    rng = random.Random(SEED)
    dict_rows = rng.sample(uniq, min(N_DICT, len(uniq)))
    print(f"辞書語彙: 漢字含み表層{len(by_surface)} 一意読み{len(uniq)} 採用{len(dict_rows)}")
    rows = []
    for fn in (CONSENSUS, DIGITBULK):
        for l in open(fn, encoding="utf-8"):
            s, label = l.rstrip("\n").split("\t", 1)
            rows.append(json.dumps({"text": s, "kana": label}, ensure_ascii=False))
    n_bulk = len(rows)
    rows.extend(AZ_ROWS)
    human = [json.dumps({"text": t, "kana": k}, ensure_ascii=False)
             for t, k in kwdlc["train"] + rohan]
    rows.extend(h for _ in range(HUMAN_OVERSAMPLE) for h in human)
    rows.extend(j for _ in range(JSUT_OVERSAMPLE) for j in JSUT_ROWS)
    rows.extend(json.dumps({"text": s, "kana": r}, ensure_ascii=False) for s, r in dict_rows)
    rows.extend(l for _ in range(NUM_OVERSAMPLE) for l in NUM_RULE_ROWS)
    rng2 = random.Random(SEED); rng2.shuffle(rows)
    open(TRAIN_FILE, "w", encoding="utf-8").write("\n".join(rows) + "\n")
    print(f"total: {len(rows)} (bulk {n_bulk} + 青空載録 {len(AZ_ROWS)} + 人手 {len(human)}x{HUMAN_OVERSAMPLE} + jsut {len(JSUT_ROWS)}x{JSUT_OVERSAMPLE} + 辞書 {len(dict_rows)} + 規則 {len(NUM_RULE_ROWS)}x{NUM_OVERSAMPLE})")
else:
    print("cached train:", sum(1 for _ in open(TRAIN_FILE, encoding="utf-8")))

In [ ]:
# ===== KWDLC評価goldの保存（評価フェーズがセッション跨ぎでも読めるように学習前に書き出す） =====
os.makedirs(f"{WORK}/results_{VER}", exist_ok=True)
json.dump(kwdlc["test"], open(f"{WORK}/results_{VER}/kwdlc_test_gold.json", "w"), ensure_ascii=False)
print("saved:", f"{WORK}/results_{VER}/kwdlc_test_gold.json", len(kwdlc["test"]))


In [ ]:
# ===== 汚染監査: 困難セット300文が訓練データに含まれないことを実行ログに記録 =====
import hashlib
_hs = set(hashlib.sha1(h["text"].encode()).hexdigest() for h in HARD_SET)

n_rows = 0; hits = []
for l in open(TRAIN_FILE, encoding="utf-8"):
    n_rows += 1
    t = json.loads(l)["text"]
    if hashlib.sha1(t.encode()).hexdigest() in _hs:
        hits.append(t)
print(f"訓練データ {n_rows} 行 × 困難セット300文: 完全一致 {len(hits)} 件")
assert not hits, f"汚染検出: {hits[:3]}"
print("汚染監査OK")

In [ ]:
# ===== 学習（チェックポイント自動再開） =====
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL)
use_bf16 = torch.cuda.is_bf16_supported()
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16 if use_bf16 else torch.float16).cuda()
from peft import LoraConfig, get_peft_model
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_R*2, target_modules="all-linear",
                                         lora_dropout=0.05, task_type="CAUSAL_LM"))
model.print_trainable_parameters()
def build_example(text, kana):
    pt = tok.apply_chat_template([{"role":"system","content":SYSTEM},{"role":"user","content":text}],
                                 tokenize=False, add_generation_prompt=True)
    pids = tok(pt, add_special_tokens=False)["input_ids"]
    tids = tok(kana + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = (pids+tids)[:MAX_LEN]
    return {"input_ids": ids, "labels": ([-100]*min(len(pids),MAX_LEN)+tids)[:MAX_LEN], "length": len(ids)}
rows = [json.loads(l) for l in open(TRAIN_FILE, encoding="utf-8")]
ds = Dataset.from_list([build_example(r["text"], r["kana"]) for r in rows])
class Collator:
    def __call__(self, feats):
        m = max(len(f["input_ids"]) for f in feats); pad = tok.pad_token_id or tok.eos_token_id
        b = {"input_ids": [], "labels": [], "attention_mask": []}
        for f in feats:
            n = m - len(f["input_ids"])
            b["input_ids"].append(f["input_ids"]+[pad]*n)
            b["labels"].append(f["labels"]+[-100]*n)
            b["attention_mask"].append([1]*len(f["input_ids"])+[0]*n)
        return {k: torch.tensor(v) for k, v in b.items()}
import inspect
ta = set(inspect.signature(TrainingArguments.__init__).parameters)
kw = dict(output_dir=f"{WORK}/ckpt_{VER}", num_train_epochs=EPOCHS, learning_rate=LR,
          per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
          bf16=use_bf16, fp16=not use_bf16, logging_steps=200, disable_tqdm=True,
          max_grad_norm=0.5, save_strategy="steps", save_steps=2000,
          save_total_limit=2, seed=SEED, report_to="none")
if "warmup_ratio" in ta: kw["warmup_ratio"] = WARMUP_RATIO
targs = TrainingArguments(**{k: v for k, v in kw.items() if k in ta})
from transformers import TrainerCallback
import time as _time
class QuietProgress(TrainerCallback):
    def __init__(self, every=2000):
        self.every = every; self.t0 = _time.time()
    def on_log(self, args, state, control, logs=None, **kw):
        pass  # 既定のログ出力は抑制
    def on_step_end(self, args, state, control, **kw):
        if state.global_step % self.every == 0 and state.global_step > 0:
            loss = next((l["loss"] for l in reversed(state.log_history) if "loss" in l), None)
            el = (_time.time() - self.t0) / 3600
            pct = 100 * state.global_step / state.max_steps if state.max_steps else 0
            eta = el / max(pct, 0.01) * (100 - pct)
            print(f"step {state.global_step}/{state.max_steps} ({pct:.1f}%)  loss={loss}  経過{el:.1f}h 残り約{eta:.1f}h", flush=True)
class SpikeGuard(TrainerCallback):
    # logged lossが閾値を超えたら学習を停止 → 外側ループが直前ckptからリトライ
    def __init__(self, thresh=2.0, warmup_steps=3000):
        self.thresh = thresh; self.warmup = warmup_steps; self.tripped = False
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs and state.global_step > self.warmup and logs["loss"] > self.thresh:
            self.tripped = True
            control.should_training_stop = True
            print(f"!! SPIKE検出 step={state.global_step} loss={logs['loss']:.2f} → 自動リトライします", flush=True)

import glob, random as _rnd
for attempt in range(6):
    if attempt > 0:
        idx = list(range(len(ds)))
        _rnd.Random(SEED + attempt).shuffle(idx)
        ds = ds.select(idx)
    guard = SpikeGuard()
    trainer = Trainer(model=model, args=targs, train_dataset=ds, data_collator=Collator(),
                      callbacks=[QuietProgress(2000), guard])
    ck = sorted(glob.glob(f"{WORK}/ckpt_{VER}/checkpoint-*"))
    print(f"attempt {attempt+1}  resume: {bool(ck)}")
    trainer.train(resume_from_checkpoint=bool(ck))
    if not guard.tripped:
        break
else:
    raise RuntimeError("スパイクが6回連続で発生。設定を見直してください")
model.save_pretrained(f"{WORK}/results_{VER}/final"); tok.save_pretrained(f"{WORK}/results_{VER}/final")
json.dump(trainer.state.log_history, open(f"{WORK}/results_{VER}/train_log.json","w"))
print("saved:", f"{WORK}/results_{VER}/final")

## 評価フェーズ（モデル単体システム: 文分割＋カナ制約＋ビーム4）

以降のセルは学習完了後に実行される。保存済みアダプタを読み直すので、
セッションが切れていてもここから再開できる。


In [ ]:
# ===== モデル読込（v11アダプタをマージ） =====
import csv, json, re, urllib.request
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList
from peft import PeftModel
MODEL = "sbintuitions/sarashina2.2-1b-instruct-v0.1"
ADAPTER = f"{WORK}/results_v11/final"
tok = AutoTokenizer.from_pretrained(ADAPTER)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="cuda")
model = PeftModel.from_pretrained(model, ADAPTER)
model = model.merge_and_unload().eval()
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
SYSTEM = ("あなたは日本語テキストの読み推定器です。与えられた文の読みを、カタカナのみで1行で出力してください。"
          "発音通りに書くこと: 助詞の「は」は「ワ」、「へ」は「エ」、「を」は「オ」。数字や記号も読みをカタカナで書くこと。")
OK = re.compile(r"^[ァ-ヶー、]+$")
allowed = sorted(set([i for i in range(len(tok)) if OK.match(tok.decode([i]))] + [tok.eos_token_id]))
allowed_t = torch.tensor(allowed, device="cuda")
print("許可トークン:", len(allowed_t), "/", len(tok))
class KanaMask(LogitsProcessor):
    def __call__(self, input_ids, scores):
        mask = torch.full_like(scores, float("-inf"))
        mask[:, allowed_t] = 0.0
        return scores + mask
_SPLIT = re.compile(r"(?<=[。！？])(?![。！？])")
def split_sentences(text):
    return [p for p in _SPLIT.split(text) if p.strip()]


In [ ]:
# ===== 生成関数（文分割＋カナマスク＋ビーム4） =====
NUM_BEAMS = 4
def gen_preds(rows, out_path, B=8):
    if os.path.exists(out_path):
        print("cached:", out_path); return
    # 文分割: (key, part_idx, part_text) に展開
    parts = []
    for k, t in rows:
        for j, p in enumerate(split_sentences(t) or [t]):
            parts.append((k, j, p))
    outs = {}
    for i in range(0, len(parts), B):
        ch = parts[i:i+B]
        prompts = [tok.apply_chat_template([{"role":"system","content":SYSTEM},{"role":"user","content":p}],
                                           tokenize=False, add_generation_prompt=True) for _,_,p in ch]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, do_sample=False, num_beams=NUM_BEAMS,
                                 length_penalty=0.0, early_stopping=True,
                                 pad_token_id=tok.pad_token_id,
                                 logits_processor=LogitsProcessorList([KanaMask()]),
                                 max_new_tokens=min(256, 4*max(len(p) for _,_,p in ch)))
        for (k, j, _), seq in zip(ch, out):
            g = tok.decode(seq[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            outs.setdefault(k, {})[j] = g.splitlines()[0] if g else ""
        if (i//B) % 100 == 0: print(out_path, i+len(ch), "/", len(parts), flush=True)
    lines = []
    for k, _ in rows:
        d = outs.get(k, {})
        lines.append(f"{k}\t" + "".join(d[j] for j in sorted(d)))
    open(out_path, "w", encoding="utf-8").write("\n".join(lines)+"\n")
    print("saved:", out_path)


In [ ]:
# ===== 評価セットの取得と生成 =====
urllib.request.urlretrieve("https://raw.githubusercontent.com/CyberAgentAILab/jvs_nonpara_kana/main/jvs_nonpara_kana.csv","jvs_nonpara_kana.csv")
jvs = [(r[0], r[1]) for r in list(csv.reader(open("jvs_nonpara_kana.csv", encoding="utf-8")))[1:]]
gen_preds(jvs, f"{WORK}/results_v11_beam/jvs3000_preds.tsv")
kwdlc_gold = json.load(open(f"{WORK}/results_v11/kwdlc_test_gold.json", encoding="utf-8"))
gen_preds([(f"kw{i:04d}", t) for i, (t, _) in enumerate(kwdlc_gold)], f"{WORK}/results_v11_beam/kwdlc_test_preds.tsv")
# 困難セット: 構築フェーズで読込済みならそれを使用（セッション再開時は同ディレクトリのCSVから）
if "HARD_SET" not in globals():
    HARD_SET_CSV = "heteronym_eval_set_v1.csv"   # このノートブックと同じディレクトリに配置（リポジトリ data/ に同梱）
    try:
        _hs_rows = list(csv.reader(open(HARD_SET_CSV, encoding="utf-8")))[1:]
    except FileNotFoundError:
        raise FileNotFoundError("heteronym_eval_set_v1.csv を作業ディレクトリにアップロードしてください（リポジトリの data/ にあります）")
    HARD_SET = [{"id": r[0], "text": r[1]} for r in _hs_rows]
    assert len(HARD_SET) == 300
gen_preds([(h["id"], h["text"]) for h in HARD_SET], f"{WORK}/results_v11_beam/hardset_preds.tsv")
G2PBENCH_URL = ("https://raw.githubusercontent.com/filmapp/ja-tts-g2p-bench/"
                "12182d464d4b5df82b829369974bf72d24a6f0cf/data/items_read_bench_v1.jsonl")
urllib.request.urlretrieve(G2PBENCH_URL, "g2pbench.jsonl")
G2PBENCH = [json.loads(l) for l in open("g2pbench.jsonl", encoding="utf-8")]
gen_preds([(it["id"], it["surface"]) for it in G2PBENCH], f"{WORK}/results_v11_beam/g2pbench_preds.tsv")
from datasets import load_dataset
joyo = load_dataset("sbintuitions/joyo-kanji-yomi-benchmark", split="train")
gen_preds([(ex["key"], ex["normalized_text"]) for ex in joyo], f"{WORK}/results_v11_beam/joyo_preds.tsv")
parakeet = load_dataset("Parakeet-Inc/joyo-kanji-yomi-benchmark-parakeet", split="test")
gen_preds([(f"pk{i:05d}", ex["text"]) for i, ex in enumerate(parakeet)], f"{WORK}/results_v11_beam/parakeet_preds.tsv")
import shutil
shutil.copy(f"{WORK}/results_v11/kwdlc_test_gold.json", f"{WORK}/results_v11_beam/")
print("all preds done")


## 実行後にやること
1. この実行済みノートブックを保存して共有
2. Driveの `g2p_run/results_v11_beam/` をダウンロード（zip）し、ローカルで採点:
   `eval/eval_jvs.py` / `eval/eval_hard_set.py` / `eval/eval_aligned_bench.py`（README参照）
